# 00 - Preparation du corpus

Ce notebook construit la base d'images du cas pratique. On rassemble quatre sources d'images de couverture, naturelles et generees, on les traite toutes de la meme facon, puis on insere un message simule avec trois algorithmes.

L'objectif est qu'a la fin, la seule difference entre deux images soit leur source ou l'insertion, jamais le traitement.

Sortie attendue : un corpus organise en `source / (cover | algorithme)`, archive sur Google Drive.

## Parametres et Drive

In [ ]:
import os, glob, random
import numpy as np

# Graine unique pour tout le projet, garantit des tirages reproductibles
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Cible retenue pour le run final. Mettre 200 pour un simple test de la chaine
N_IMAGES = 1500
IMG_SIZE = 256
PAYLOADS = [0.2, 0.4]      # charges utiles testees, en bits par pixel
ALGOS = ['lsb', 'uniward', 'hill']

ROOT = '/content/corpus'  # dossier de travail local a la session Colab
os.makedirs(ROOT, exist_ok=True)
print('Parametres charges,', N_IMAGES, 'images par source.')

In [ ]:
# On monte Drive pour sauvegarder le corpus en fin de notebook
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/memoire_data'
    os.makedirs(DRIVE, exist_ok=True)
except Exception as e:
    DRIVE = None
    print('Drive non monte (execution hors Colab).', e)

## Installation

In [ ]:
# conseal insere les messages, les autres servent au telechargement et au traitement d'images
!pip install -q conseal huggingface_hub datasets imageio pillow
print('Installation terminee.')

Le telechargement Hugging Face est lent quand la requete est anonyme. Pour l'accelerer, vous pouvez coller un jeton gratuit (huggingface.co, section Access Tokens) dans la cellule suivante. Sinon, laissez la telle quelle.

In [ ]:
# Optionnel : un jeton HF accelere les telechargements
HF_TOKEN = ''   # collez votre jeton entre les guillemets si vous en avez un
if HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)
    print('Jeton HF actif.')
else:
    print('Sans jeton, telechargements plus lents mais fonctionnels.')

## 1. Fonctions communes

Le meme pretraitement et la meme insertion pour toutes les sources. C'est ce qui rend les comparaisons valables.

In [ ]:
import imageio.v2 as imageio
import conseal as cl
from PIL import Image

def preprocess(src_files, dst_dir, n):
    # Convertit en gris, recadre au centre, sauve en PGM sans perte
    os.makedirs(dst_dir, exist_ok=True)
    saved = []
    for path in src_files:
        if len(saved) >= n:
            break
        try:
            im = Image.open(path).convert('L')
            w, h = im.size
            if min(w, h) < IMG_SIZE:   # trop petite pour un recadrage propre
                continue
            left, top = (w - IMG_SIZE) // 2, (h - IMG_SIZE) // 2
            im = im.crop((left, top, left + IMG_SIZE, top + IMG_SIZE))
            out = f'{dst_dir}/{len(saved):04d}.pgm'
            im.save(out)
            saved.append(out)
        except Exception:
            pass
    return saved

def embed_one(x, algo, payload, seed):
    # API conseal : image en x0, charge utile en alpha
    if algo == 'lsb':
        return cl.lsb.simulate(x0=x, alpha=payload, seed=seed)
    if algo == 'uniward':
        return cl.suniward.simulate_single_channel(x0=x, alpha=payload, seed=seed)
    if algo == 'hill':
        return cl.hill.simulate_single_channel(x0=x, alpha=payload, seed=seed)
    raise ValueError(algo)

def embed_folder(cover_paths, algo, payload, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    for p in cover_paths:
        x = imageio.imread(p).astype(np.uint8)
        # graine derivee du nom de fichier, pour un tirage stable par image
        s = int(np.random.default_rng(SEED + hash(p) % 10000).integers(1e6))
        y = embed_one(x, algo, payload, s)
        base = os.path.basename(p).replace('.pgm', '')
        imageio.imwrite(f'{dst_dir}/{base}_p{payload}.pgm', y.astype(np.uint8))

print('Fonctions pretes.')

## 2. Telechargement des images de couverture

Quatre sources. Les naturelles viennent de BOSSBase, les generees de jeux publics deja produits par des modeles de diffusion. On ne genere rien, on telecharge, et on ne prend que ce dont on a besoin.

In [ ]:
import urllib.request, zipfile, shutil
from huggingface_hub import hf_hub_download
from datasets import load_dataset

def get_bossbase(dst, n):
    # Images naturelles de reference. Archive volumineuse, telechargee une seule fois
    os.makedirs(dst, exist_ok=True)
    url = 'http://dde.binghamton.edu/download/ImageDB/BOSSbase_1.01.zip'
    if not glob.glob(f'{dst}/**/*.pgm', recursive=True):
        try:
            urllib.request.urlretrieve(url, '/content/boss.zip')
            with zipfile.ZipFile('/content/boss.zip') as z:
                z.extractall(dst)
        except Exception as e:
            print('Telechargement BOSSBase impossible, televersez des .pgm dans', dst, ':', e)
    return sorted(glob.glob(f'{dst}/**/*.pgm', recursive=True))[:n]

def get_diffusiondb(dst, n):
    # Stable Diffusion 1.x. Une archive contient 1000 images, on en prend assez pour atteindre n
    os.makedirs(dst, exist_ok=True)
    part = 1
    while len(glob.glob(f'{dst}/**/*.png', recursive=True)) < n and part <= 6:
        zp = hf_hub_download('poloclub/diffusiondb', repo_type='dataset',
                             filename=f'images/part-{part:06d}.zip')
        with zipfile.ZipFile(zp) as z:
            z.extractall(dst)
        part += 1
    return sorted(glob.glob(f'{dst}/**/*.png', recursive=True))[:n]

def get_hf_stream(repo, dst, n):
    # Streaming : recupere seulement les n premieres images, sans tout telecharger
    os.makedirs(dst, exist_ok=True)
    ds = load_dataset(repo, split='train', streaming=True)
    col, cnt = None, 0
    for row in ds:
        if col is None:
            col = next((c for c in row if c.lower() in ('image','img','png','jpg')), list(row)[0])
        if cnt >= n:
            break
        try:
            row[col].save(f'{dst}/img_{cnt:05d}.png'); cnt += 1
            if cnt % 200 == 0:
                print(f'  {repo}: {cnt}/{n}')
        except Exception:
            pass
    return sorted(glob.glob(f'{dst}/*.png'))[:n]

def get_genimage_generator(repo, gen_name, dst, n):
    # Recupere uniquement les images d'un generateur precis de GenImage, en streaming
    os.makedirs(dst, exist_ok=True)
    ds = load_dataset(repo, split='train', streaming=True)
    names = ds.features['generator'].names
    idx = [i for i, x in enumerate(names) if gen_name.lower() in x.lower()][0]
    cnt = 0
    for row in ds:
        if cnt >= n:
            break
        if row['generator'] == idx:
            row['image'].save(f'{dst}/img_{cnt:05d}.png'); cnt += 1
            if cnt % 200 == 0:
                print(f'  {gen_name}: {cnt}/{n}')
    return sorted(glob.glob(f'{dst}/*.png'))[:n]

print('Fonctions de telechargement pretes.')

In [ ]:
# On telecharge source par source, avec un repere pour suivre la progression
print('1/4 BOSSBase (archive volumineuse, longue la premiere fois)')
raw_nat = get_bossbase('/content/raw_natural', N_IMAGES); print('  ok', len(raw_nat))

print('2/4 Stable Diffusion')
raw_sd = get_diffusiondb('/content/raw_sd', N_IMAGES); print('  ok', len(raw_sd))

print('3/4 SDXL')
raw_sdxl = get_hf_stream('ostris/sdxl_10_reg', '/content/raw_sdxl', N_IMAGES); print('  ok', len(raw_sdxl))

print('4/4 ADM (filtre sur le bon generateur)')
raw_adm = get_genimage_generator('TheKernel01/Tiny-GenImage', 'ADM', '/content/raw_adm', N_IMAGES); print('  ok', len(raw_adm))

In [ ]:
# On pretraite chaque source vers corpus/<source>/cover
covers = {}
covers['natural'] = preprocess(raw_nat,  f'{ROOT}/natural/cover', N_IMAGES)
covers['sd']      = preprocess(raw_sd,   f'{ROOT}/sd/cover',      N_IMAGES)
covers['sdxl']    = preprocess(raw_sdxl, f'{ROOT}/sdxl/cover',    N_IMAGES)
covers['adm']     = preprocess(raw_adm,  f'{ROOT}/adm/cover',     N_IMAGES)
for src, lst in covers.items():
    print(f'{src:10s}: {len(lst)} covers')

## 3. Insertion des messages

Pour chaque source, on applique les trois algorithmes aux deux charges utiles. On obtient les versions porteuses de chaque cover. A 1500 images cette etape prend du temps, c'est normal.

In [ ]:
for src, cover_list in covers.items():
    if not cover_list:
        print('source vide, ignoree :', src)
        continue
    for algo in ALGOS:
        for payload in PAYLOADS:
            embed_folder(cover_list, algo, payload, f'{ROOT}/{src}/{algo}')
    print(f'{src:10s}: insertion des 3 algorithmes terminee')

## 4. Controle rapide

Deux verifications avant de sauvegarder. D'abord que l'insertion reste invisible a l'oeil. Ensuite le kurtosis des residus par source, qui doit montrer quatre empreintes distinctes.

In [ ]:
import matplotlib.pyplot as plt

cov = imageio.imread(covers['natural'][0]).astype(float)
base = os.path.basename(covers['natural'][0]).replace('.pgm','')
stg = imageio.imread(f'{ROOT}/natural/uniward/{base}_p0.4.pgm').astype(float)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(cov, cmap='gray'); ax[0].set_title('cover'); ax[0].axis('off')
ax[1].imshow(stg, cmap='gray'); ax[1].set_title('porteuse'); ax[1].axis('off')
ax[2].imshow(np.abs(stg-cov), cmap='hot'); ax[2].set_title('difference'); ax[2].axis('off')
plt.tight_layout(); plt.show()
print('pixels modifies :', int(np.sum(stg != cov)), 'sur', cov.size)

In [ ]:
from scipy.stats import kurtosis
from scipy.signal import convolve2d

def residu_hf(img):
    # filtre passe haut simple, fait ressortir les traces d'insertion
    k = np.array([[-1,2,-1],[2,-4,2],[-1,2,-1]], float)
    return convolve2d(img, k, mode='same', boundary='symm')

def kurt_moyen(paths):
    vals = [kurtosis(residu_hf(imageio.imread(p).astype(float)).ravel()) for p in paths[:80]]
    return float(np.mean(vals)) if vals else float('nan')

print('Kurtosis moyen des residus, cover vierge par source :')
for src in ['natural', 'sd', 'sdxl', 'adm']:
    if covers.get(src):
        print(f'  {src:10s}: {kurt_moyen(covers[src]):.2f}')
print('\nQuatre valeurs bien distinctes confirment quatre domaines statistiques differents.')

## 5. Sauvegarde du corpus

On archive sur Drive dans un dossier date. Un ancien instantane n'est jamais ecrase.

In [ ]:
from datetime import date

print('=== Etat du corpus ===')
for src in ['natural', 'sd', 'sdxl', 'adm']:
    ligne = f'  {src:10s}: '
    for sub in ['cover'] + ALGOS:
        ligne += f'{sub}={len(glob.glob(f"{ROOT}/{src}/{sub}/*.pgm"))}  '
    print(ligne)

if DRIVE:
    nom = f'corpus_{date.today().isoformat()}_n{N_IMAGES}'
    shutil.make_archive(f'{DRIVE}/{nom}', 'zip', ROOT)
    print('Corpus sauvegarde :', f'{DRIVE}/{nom}.zip')
else:
    print('Drive non disponible, corpus seulement en local.')

## Suite

Le corpus est pret. Prochain notebook : `01_extraction_features`, qui extrait les caracteristiques SRM et les met en cache pour ne pas les recalculer.

Pensez a enregistrer ce notebook sur GitHub et a noter l'etape dans le README.